In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("datasets/survey_results_public.csv")
df2 = df.copy()

In [3]:
# print(df2.head())
# print(df2.shape)
print(df2.columns)
df2.info()

Index(['Respondent', 'MainBranch', 'Hobbyist', 'OpenSourcer', 'OpenSource',
       'Employment', 'Country', 'Student', 'EdLevel', 'UndergradMajor',
       'EduOther', 'OrgSize', 'DevType', 'YearsCode', 'Age1stCode',
       'YearsCodePro', 'CareerSat', 'JobSat', 'MgrIdiot', 'MgrMoney',
       'MgrWant', 'JobSeek', 'LastHireDate', 'LastInt', 'FizzBuzz',
       'JobFactors', 'ResumeUpdate', 'CurrencySymbol', 'CurrencyDesc',
       'CompTotal', 'CompFreq', 'ConvertedComp', 'WorkWeekHrs', 'WorkPlan',
       'WorkChallenge', 'WorkRemote', 'WorkLoc', 'ImpSyn', 'CodeRev',
       'CodeRevHrs', 'UnitTests', 'PurchaseHow', 'PurchaseWhat',
       'LanguageWorkedWith', 'LanguageDesireNextYear', 'DatabaseWorkedWith',
       'DatabaseDesireNextYear', 'PlatformWorkedWith',
       'PlatformDesireNextYear', 'WebFrameWorkedWith',
       'WebFrameDesireNextYear', 'MiscTechWorkedWith',
       'MiscTechDesireNextYear', 'DevEnviron', 'OpSys', 'Containers',
       'BlockchainOrg', 'BlockchainIs', 'BetterLife'

In [4]:
var = "gender"
if var in df2.columns.str.lower():
    print(f"{var} column exists")
else:
    print(f"{var} column does not exist")

gender column does not exist


### 1. The average age of developers when they wrote their first line of code

In [5]:
df2["Age1stCode"] = pd.to_numeric(df2["Age1stCode"], errors="coerce")

average_age = df2["Age1stCode"].mean()

# df2["Age1stCode"] = df2["Age1stCode"].fillna(average_age)

# df2.info()
print(df2["Age1stCode"].dtype)
print(df2["Age1stCode"].isnull().sum())
print("Average age:", average_age)

float64
1795
Average age: 15.456446353114092


### 2. The percentage of developers who knew Python, per country

In [6]:
df2[["Country", "LanguageWorkedWith"]].head(10)

,Country,LanguageWorkedWith
0,United Kingdom,HTML/CSS;Java;JavaScript;Python
1,Bosnia and Herzegovina,C++;HTML/CSS;Python
2,Thailand,HTML/CSS
3,United States,C;C++;C#;Python;SQL
4,Ukraine,C++;HTML/CSS;Java;JavaScript;Python;SQL;VBA
5,Canada,Java;R;SQL
6,Ukraine,HTML/CSS;JavaScript
7,India,Bash/Shell/PowerShell;C;C++;HTML/CSS;Java;Java...
8,New Zealand,Bash/Shell/PowerShell;C#;HTML/CSS;JavaScript;P...
9,India,C#;Go;JavaScript;Python;R;SQL


In [7]:
df2["LanguageWorkedWith"] = df2["LanguageWorkedWith"].str.split(";")
# print(df2["LanguageWorkedWith"])
df2 = df2.explode("LanguageWorkedWith")
print(df2[["Country", "LanguageWorkedWith"]])

                      Country LanguageWorkedWith
0              United Kingdom           HTML/CSS
0              United Kingdom               Java
0              United Kingdom         JavaScript
0              United Kingdom             Python
1      Bosnia and Herzegovina                C++
...                       ...                ...
88882                   Spain               Java
88882                   Spain         JavaScript
88882                   Spain                PHP
88882                   Spain                SQL
88882                   Spain              Swift

[441819 rows x 2 columns]


In [8]:
total = df2.groupby("Country")["Respondent"].nunique()
print(total)

Country
Afghanistan                              44
Albania                                  86
Algeria                                 134
Andorra                                   7
Angola                                    5
                                       ... 
Venezuela, Bolivarian Republic of...     88
Viet Nam                                231
Yemen                                    19
Zambia                                   12
Zimbabwe                                 39
Name: Respondent, Length: 179, dtype: int64


In [9]:
python_count = df2[df2["LanguageWorkedWith"] == "Python"].groupby("Country")["Respondent"].nunique()
print(python_count)

Country
Afghanistan                               8
Albania                                  23
Algeria                                  40
Angola                                    2
Argentina                               191
                                       ... 
Venezuela, Bolivarian Republic of...     28
Viet Nam                                 78
Yemen                                     3
Zambia                                    4
Zimbabwe                                 14
Name: Respondent, Length: 162, dtype: int64


In [10]:
python_percentage = (python_count / total) * 100
print(python_percentage)

Country
Afghanistan                             18.181818
Albania                                 26.744186
Algeria                                 29.850746
Andorra                                       NaN
Angola                                  40.000000
                                          ...    
Venezuela, Bolivarian Republic of...    31.818182
Viet Nam                                33.766234
Yemen                                   15.789474
Zambia                                  33.333333
Zimbabwe                                35.897436
Name: Respondent, Length: 179, dtype: float64


### 3. Average salary of developers by continent

In [11]:
# !pip install pycountry-convert
import pycountry_convert as pc

In [12]:
df2 = df[["Country", "ConvertedComp"]]
df2.head()

,Country,ConvertedComp
0,United Kingdom,NaN
1,Bosnia and Herzegovina,NaN
2,Thailand,8820.0
3,United States,61000.0
4,Ukraine,NaN


In [13]:
country_code = pc.country_name_to_country_alpha2("India")
print(country_code)
continent_code = pc.country_alpha2_to_continent_code(country_code)
print(continent_code)
continent_name = pc.convert_continent_code_to_continent_name(continent_code)
print(continent_name)

IN
AS
Asia


In [14]:
def get_continent(country):
    try:
        country_code = pc.country_name_to_country_alpha2(country)
        continent_code = pc.country_alpha2_to_continent_code(country_code)
        return pc.convert_continent_code_to_continent_name(continent_code)
    except:
        return None

In [15]:
df2["Continent"] = df2["Country"].apply(get_continent)

In [16]:
print(df2.head(10))

                  Country  ConvertedComp      Continent
0          United Kingdom            NaN         Europe
1  Bosnia and Herzegovina            NaN         Europe
2                Thailand         8820.0           Asia
3           United States        61000.0  North America
4                 Ukraine            NaN         Europe
5                  Canada       366420.0  North America
6                 Ukraine            NaN         Europe
7                   India            NaN           Asia
8             New Zealand        95179.0        Oceania
9                   India        13293.0           Asia


In [17]:
initial_average_salary = df2.groupby("Continent")["ConvertedComp"].mean()
print(initial_average_salary)

Continent
Africa            34925.546400
Asia              36476.492550
Europe            97116.182129
North America    225667.247418
Oceania          166950.955791
South America     33010.825072
Name: ConvertedComp, dtype: float64


In [18]:
unmapped = df2[df2["Continent"].isna()]["Country"].unique()
print(unmapped)

<ArrowStringArray>
[                       'Hong Kong (S.A.R.)',
                    'Libyan Arab Jamahiriya',
      'Venezuela, Bolivarian Republic of...',
          'Other Country (Not Listed Above)',
 'The former Yugoslav Republic of Macedonia',
                         'Republic of Korea',
                 'Congo, Republic of the...',
                               'Timor-Leste',
                                         nan]
Length: 9, dtype: str


In [19]:
print("Total unique countries:", df["Country"].nunique())
print("Mapped countries:", df2.dropna(subset=["Continent"])["Country"].nunique())
print("Unmapped countries:", len(unmapped))

Total unique countries: 179
Mapped countries: 171
Unmapped countries: 9


In [20]:
print("Missing country values:", df2["Country"].isna().sum())

Missing country values: 132


In [21]:
df2[df2["Continent"].isna()]["Country"].value_counts(dropna=False)

Country
Hong Kong (S.A.R.)                           188
Other Country (Not Listed Above)             136
NaN                                          132
Venezuela, Bolivarian Republic of...          88
The former Yugoslav Republic of Macedonia     54
Republic of Korea                             39
Libyan Arab Jamahiriya                        10
Congo, Republic of the...                      7
Timor-Leste                                    1
Name: count, dtype: int64

In [22]:
def get_continent(country):
    if country == "Timor-Leste":
        return "Asia"
    country_mapping = {
        "Hong Kong (S.A.R.)": "Hong Kong",
        "Venezuela, Bolivarian Republic of...": "Venezuela",
        "The former Yugoslav Republic of Macedonia": "North Macedonia",
        "Republic of Korea": "South Korea",
        "Libyan Arab Jamahiriya": "Libya",
        "Congo, Republic of the...": "Republic of the Congo",
    }

    try:
        country = country_mapping.get(country, country)

        country_code = pc.country_name_to_country_alpha2(country)
        continent_code = pc.country_alpha2_to_continent_code(country_code)

        return pc.convert_continent_code_to_continent_name(continent_code)

    except:
        return None

In [23]:
df2["Continent"] = df2["Country"].apply(get_continent)

In [24]:
final_average_salary = df2.groupby("Continent")["ConvertedComp"].mean()
print(initial_average_salary)
print("\n\n",final_average_salary)

Continent
Africa            34925.546400
Asia              36476.492550
Europe            97116.182129
North America    225667.247418
Oceania          166950.955791
South America     33010.825072
Name: ConvertedComp, dtype: float64


 Continent
Africa            35045.782816
Asia              37768.549543
Europe            97042.157518
North America    225667.247418
Oceania          166950.955791
South America     32691.801127
Name: ConvertedComp, dtype: float64


### 4. The most desired programming language for 2020

In [25]:
df2 = df[["LanguageDesireNextYear"]]
df2["LanguageDesireNextYear"].head(10)

0      C;C++;C#;Go;HTML/CSS;Java;JavaScript;Python;SQL
1                          C++;HTML/CSS;JavaScript;SQL
2                                      Elixir;HTML/CSS
3                                  C;C#;JavaScript;SQL
4             HTML/CSS;Java;JavaScript;SQL;WebAssembly
5                                     Python;Scala;SQL
6                                  HTML/CSS;JavaScript
7    Bash/Shell/PowerShell;C;C++;Elixir;Erlang;Go;P...
8    Bash/Shell/PowerShell;C;HTML/CSS;JavaScript;Ru...
9                 C#;Go;JavaScript;Kotlin;Python;R;SQL
Name: LanguageDesireNextYear, dtype: str

In [26]:
df2["LanguageDesireNextYear"] = df2["LanguageDesireNextYear"].str.split(";")

df2 = df2.explode("LanguageDesireNextYear")

language_count = df2["LanguageDesireNextYear"].value_counts()

print(language_count)
print("\nMost desired language:", language_count.idxmax())

LanguageDesireNextYear
JavaScript               44739
Python                   40006
HTML/CSS                 36645
SQL                      33566
TypeScript               23720
Java                     23508
C#                       22449
Bash/Shell/PowerShell    21303
Go                       17060
C++                      16856
Kotlin                   13261
PHP                      12837
C                        11143
Rust                     10497
Swift                     8800
WebAssembly               8440
Other(s):                 7912
Ruby                      7251
R                         6734
Scala                     5562
Assembly                  4659
Dart                      4429
Elixir                    3788
F#                        3521
Objective-C               2851
Clojure                   2752
Erlang                    2124
VBA                       1641
Name: count, dtype: int64

Most desired language: JavaScript


### 5. People who code as a hobby(groupby continent only)

In [27]:
df2 = df[["Country", "Hobbyist"]]
df2["Continent"] = df2["Country"].apply(get_continent)

In [28]:
df2["Hobbyist"].value_counts()

Hobbyist
Yes    71257
No     17626
Name: count, dtype: int64

In [29]:
hobby_report = df2.groupby(["Continent", "Hobbyist"]).size()
print(hobby_report)

Continent      Hobbyist
Africa         No            586
               Yes          2163
Asia           No           4071
               Yes         14502
Europe         No           6440
               Yes         29403
North America  No           5084
               Yes         20410
Oceania        No            421
               Yes          2013
South America  No            855
               Yes          2667
dtype: int64


In [30]:
hobby_percentage = (
    valid[valid["Hobbyist"] == "Yes"]
    .groupby("Continent").size()
    / valid.groupby("Continent").size()
) * 100

print(hobby_percentage)

NameError: name 'valid' is not defined

### 6. Job and career satisfaction(groupby continent only)

In [ ]:
df2 = df[["Country", "JobSat", "CareerSat"]]
df2["Continent"] = df2["Country"].apply(get_continent)

In [ ]:
print(df2["JobSat"].value_counts(dropna=False))
print("\n",df2["CareerSat"].value_counts(dropna=False))

In [ ]:
valid_job = df2.dropna(subset=["JobSat"])
job_report = valid_job.groupby(["Continent", "JobSat"]).size()

print(job_report)

In [ ]:
valid_career = df2.dropna(subset=["CareerSat"])
career_report = valid_career.groupby(["Continent", "CareerSat"]).size()

print(career_report)